# Chapter 34
## Nested Gamma Theta Rhythms
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

A_CURRENT and PRE_OLM_X_INF_TAU_X are static plots of gating-variable
curves (no ODE simulation) -- Brian2 has nothing to add there. This
notebook covers the three single-neuron sub-examples (the network
sub-examples EIO_1/PING_WITH_THETA_DRIVE/PING_WITH_THETA_INHIBITION are
substantially larger 40-50-neuron networks and aren't covered here).

In [ ]:
import brian2 as b2
import matplotlib.pyplot as plt
import numpy as np

### Figure 34.5
Pre-OLM Cell Voltage Trace

The pre-OLM cell is a WB-style Na/K neuron with C=1.3uF/cm^2 (not the
usual 1uF/cm^2 -- the book's own MATLAB script divides by `c` here,
unlike most other chapters where c=1 makes that division a no-op).

In [ ]:
def simulate_pre_OLM_neuron(i_ext, simulation_time, dt=0.01 * b2.ms):
    El = -70 * b2.mV
    EK = -100 * b2.mV
    ENa = 90 * b2.mV
    gl = 0.05 * b2.msiemens
    gK = 23 * b2.msiemens
    gNa = 30 * b2.msiemens
    C = 1.3 * b2.ufarad

    eqs = """
    I_e : amp

    alpham = (vm + 38*mV) / (10*mV) / (1.0 - exp(-(vm + 38*mV) / (10*mV))) /ms : Hz
    alphah = 0.07 * exp(-(vm + 63.0*mV) / (20.0*mV))/ms : Hz
    alphan = 0.018/mV * (vm - 25*mV) / (1.0 - exp(-(vm - 25*mV) / (25*mV)))/ms : Hz

    betam = 4.0 * exp(-(vm + 65.0*mV) / (18.0*mV))/ms : Hz
    betah = 1.0 / (exp(-(vm + 33.0*mV) / (10.0*mV)) + 1.0)/ms : Hz
    betan = 0.0036/mV * (35*mV - vm) / (1.0 - exp(-(35*mV - vm) / (12*mV)))/ms : Hz

    m = alpham / (alpham + betam) : 1
    membrane_Im = I_e + gNa*m**3*h*(ENa-vm) + \
        gl*(El-vm) + gK*n**4*(EK-vm) : amp

    dn/dt = alphan*(1-n)-betan*n : 1
    dh/dt = alphah*(1-h)-betah*h : 1
    dvm/dt = membrane_Im/C : volt
    """

    neuron = b2.NeuronGroup(1, eqs, method="rk4", dt=dt)
    neuron.vm = -63 * b2.mV
    neuron.I_e = i_ext
    neuron.h = "alphah / (alphah + betah)"
    neuron.n = "alphan / (alphan + betan)"

    st_mon = b2.StateMonitor(neuron, "vm", record=True)
    net = b2.Network(neuron, st_mon)
    net.run(simulation_time)
    return st_mon


sm_pre = simulate_pre_OLM_neuron(1.5 * b2.uA, 200 * b2.ms)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(sm_pre.t / b2.ms, sm_pre.vm[0] / b2.mV, lw=2, c="k")
ax.set_xlim(0, 200)
ax.set_xlabel("time [ms]")
ax.set_ylabel("v [mV]")
plt.tight_layout()
plt.show()

### The OLM Cell (with h-current, optionally with A-current too)

Adds an h-current (`r`) and, optionally, an A-current (`a`, `b`) to the
pre-OLM model above.

In [ ]:
def simulate_OLM_neuron(i_ext, simulation_time, g_h=12 * b2.msiemens, g_A=0 * b2.msiemens,
                         dt=0.01 * b2.ms):
    El = -70 * b2.mV
    EK = -100 * b2.mV
    ENa = 90 * b2.mV
    EH = -32.9 * b2.mV
    EA = -90 * b2.mV
    gl = 0.05 * b2.msiemens
    gK = 23 * b2.msiemens
    gNa = 30 * b2.msiemens
    C = 1.3 * b2.ufarad

    eqs = """
    I_e : amp

    alpham = (vm + 38*mV) / (10*mV) / (1.0 - exp(-(vm + 38*mV) / (10*mV))) /ms : Hz
    alphah = 0.07 * exp(-(vm + 63.0*mV) / (20.0*mV))/ms : Hz
    alphan = 0.018/mV * (vm - 25*mV) / (1.0 - exp(-(vm - 25*mV) / (25*mV)))/ms : Hz

    betam = 4.0 * exp(-(vm + 65.0*mV) / (18.0*mV))/ms : Hz
    betah = 1.0 / (exp(-(vm + 33.0*mV) / (10.0*mV)) + 1.0)/ms : Hz
    betan = 0.0036/mV * (35*mV - vm) / (1.0 - exp(-(35*mV - vm) / (12*mV)))/ms : Hz

    r_inf = 1.0 / (1.0 + exp((vm + 84.0*mV) / (10.2*mV))) : 1
    tau_r = 1/(exp(-14.59 - 0.086*vm/mV) + exp(-1.87 + 0.0701*vm/mV))*ms : second

    a_inf = 1.0 / (1.0 + exp(-(vm + 14.0*mV) / (16.6*mV))) : 1
    b_inf = 1.0 / (1.0 + exp((vm + 71.0*mV) / (7.3*mV))) : 1
    tau_a = 5*ms : second
    tau_b = 1/(0.000009/exp((vm/mV - 26)/28.5) + 0.014/(0.2 + exp(-(vm/mV + 70.0)/11.0)))*ms : second

    m = alpham / (alpham + betam) : 1
    membrane_Im = I_e + gNa*m**3*h*(ENa-vm) + gl*(El-vm) + gK*n**4*(EK-vm) \
        + g_h*r*(EH-vm) + g_A*a*b*(EA-vm) : amp

    dn/dt = alphan*(1-n)-betan*n : 1
    dh/dt = alphah*(1-h)-betah*h : 1
    dr/dt = (r_inf - r) / tau_r : 1
    da/dt = (a_inf - a) / tau_a : 1
    db/dt = (b_inf - b) / tau_b : 1
    dvm/dt = membrane_Im/C : volt
    """

    neuron = b2.NeuronGroup(1, eqs, method="rk4", dt=dt,
                             namespace={"g_h": g_h, "g_A": g_A})
    neuron.vm = -63 * b2.mV
    neuron.I_e = i_ext
    neuron.h = "alphah / (alphah + betah)"
    neuron.n = "alphan / (alphan + betan)"
    neuron.r = "r_inf"
    neuron.a = "a_inf"
    neuron.b = "b_inf"

    st_mon = b2.StateMonitor(neuron, ["vm", "r", "a", "b"], record=True)
    net = b2.Network(neuron, st_mon)
    net.run(simulation_time)
    return st_mon

### Figure 34.5 (OLM With h-Current)

In [ ]:
sm_h = simulate_OLM_neuron(0 * b2.uA, 200 * b2.ms, g_h=12 * b2.msiemens, g_A=0 * b2.msiemens)

fig, ax = plt.subplots(2, figsize=(7, 4), sharex=True)
ax[0].plot(sm_h.t / b2.ms, sm_h.vm[0] / b2.mV, lw=2, c="k")
ax[0].set_ylabel("v [mV]")
ax[1].plot(sm_h.t / b2.ms, sm_h.r[0], lw=2, c="k")
ax[1].set_ylim(0, 0.01)
ax[1].set_xlabel("time [ms]")
ax[1].set_ylabel("r")
plt.tight_layout()
plt.show()

### Figure 34.8 (OLM With h- and A-Currents)

In [ ]:
sm_ha = simulate_OLM_neuron(0 * b2.uA, 500 * b2.ms, g_h=12 * b2.msiemens, g_A=22 * b2.msiemens)

fig, ax = plt.subplots(2, figsize=(7, 4), sharex=True)
ax[0].plot(sm_ha.t / b2.ms, sm_ha.vm[0] / b2.mV, lw=2, c="k")
ax[0].set_ylabel("v [mV]")
ax[1].plot(sm_ha.t / b2.ms, sm_ha.a[0] * sm_ha.b[0], lw=2, c="k")
ax[1].set_ylim(0, 0.05)
ax[1].set_xlabel("time [ms]")
ax[1].set_ylabel("ab")
plt.tight_layout()
plt.show()

### PING Network with a Theta Rhythm

40 RTM E-cells + 10 WB I-cells (no E-E coupling), all-to-all E->I, I->E
and I->I via the usual q,s synaptic cascade. WB I-cells use an extra
"phi=5" speed-up on their h and n gating kinetics (the book's own
`tau_h_i`/`tau_n_i` divide by `phi`), unlike the plain WB cells used
elsewhere in this project.

Theta enters one of two ways (the two sub-examples share everything
else): `mode="drive"` sinusoidally modulates the E-cells' external
drive; `mode="inhibition"` instead adds a periodic inhibitory
conductance directly onto the E-cells.

E-cell initial conditions are on the limit cycle at 40 random phases
(computed once via a plain `scipy.odeint` run at the population's mean
drive, same warm-up idea as chapter 24's splay states -- with actual
per-cell drive noise and network coupling, the point is a
desynchronized start, not bit-exact phases). I-cells start at rest.

In [ ]:
IC_V = [-77.0381137848792, -58.19763891925973, -65.32364678823043, -68.88405048446194, -88.9824524629159, -88.98405459611014, -96.02986774971967, -61.972281935786384, -68.81165469440344, -65.91719106562044, -97.33815246987918, -54.71931422984848, -62.87286781346942, -85.42938535143489, -87.30852592995072, -87.208634315897, -80.36902029427418, -71.21484305750462, -74.62098461791453, -81.034445618317, -68.49873142897333, -90.09690300563605, -80.98710654979534, -77.4037805052307, -73.67905231455774, -64.03553851694817, -86.19652243289792, -71.57256174249142, -69.06944492183837, -96.83890254101028, -68.6235926278955, -88.03175776265209, -95.5143344643152, -58.363273259896204, -56.05263467019767, -63.471784634173915, -80.35025598314208, -93.07098637838168, -66.52435689378832, -74.29582586681117]
IC_H = [0.999287960374372, 0.988676719158269, 0.9970166534073532, 0.9987259496809486, 0.9860392172684393, 0.9860322824498475, 0.8367147701921386, 0.9935170549394433, 0.9987052064783644, 0.9974219729105225, 0.45063250412228606, 0.9858962156894256, 0.994652932876401, 0.9951045852498632, 0.9915890332158493, 0.9918334821129713, 0.9986679170833008, 0.9991970073872416, 0.9994036902420442, 0.9984449493632022, 0.9986103865750497, 0.9801700331160553, 0.9984623091680113, 0.9992501358960539, 0.999391270846939, 0.9959180103073577, 0.9939151382125339, 0.9992415184616151, 0.9987771130113493, 0.7652939434534527, 0.9986492463609842, 0.9895610025183279, 0.8675622509607565, 0.9888554814727635, 0.9867622615718226, 0.9953363926603771, 0.998673584749897, 0.9463314436656098, 0.9977785877148294, 0.9994029733100942]
IC_N = [0.004597172915739794, 0.0883735114730267, 0.03652020503140099, 0.019427653645558006, 0.020680477011609584, 0.020690406828094317, 0.18233318236044277, 0.06129000909078426, 0.019685447057233404, 0.0329827983407513, 0.46811526008578097, 0.10213782358885122, 0.054040466423200345, 0.007511025999077086, 0.012621912657172475, 0.012263575973187769, 0.003503914926885853, 0.012663537785934447, 0.00681005341831527, 0.0035599733692339764, 0.020837329959957896, 0.028936486206165204, 0.0035525753783886287, 0.004370840967505557, 0.008058159158063103, 0.04519441156595958, 0.009221086585418174, 0.011854862538001176, 0.018781994747842302, 0.24310999127000296, 0.02037030048418947, 0.015588655939663291, 0.15391986238899694, 0.08744203497558173, 0.09803353112584373, 0.04938350511092764, 0.0035037861647308, 0.07174265660975107, 0.029666153664650172, 0.007213540561787385]


def simulate_PING_with_theta(mode, simulation_time, num_e=40, num_i=10,
                              i_ext_e_mean=1.4, sigma_e=0.05,
                              g_hat_ei=0.25, g_hat_ie=0.25, g_hat_ii=0.25,
                              P=125 * b2.ms, alpha=0.8, g_ex=0.2 * b2.msiemens,
                              tau_r_e=0.5 * b2.ms, tau_d_e=3 * b2.ms, tau_dq_e=0.17229 * b2.ms,
                              tau_r_i=0.5 * b2.ms, tau_d_i=9 * b2.ms, tau_dq_i=0.11629 * b2.ms,
                              seed=42, dt=0.01 * b2.ms):
    assert mode in ("drive", "inhibition")
    rng = np.random.default_rng(seed)

    # --- E population (RTM) ---
    El_e, EK_e, ENa_e = -67 * b2.mV, -100 * b2.mV, 50 * b2.mV
    gl_e, gK_e, gNa_e = 0.1 * b2.msiemens, 80 * b2.msiemens, 100 * b2.msiemens
    C = 1 * b2.ufarad
    v_rev_e, v_rev_i = 0 * b2.mV, -75 * b2.mV

    if mode == "drive":
        i_e_line = "I_e = i_ext_e*(1 + alpha*sin(2*pi*t/P)) : amp\n    i_ext_e : amp"
        iinh_line = "Iinh = 0*amp : amp"
    else:
        i_e_line = "I_e : amp"
        iinh_line = "Iinh = g_ex*exp(-10*sin(pi*t/P)**2)*(v_rev_i - vm) : amp"

    eqs_e = """
    """ + i_e_line + """
    """ + iinh_line + """
    Isyn : amp

    alphah = 0.128 * exp(-(vm + 50.0*mV) / (18.0*mV))/ms :Hz
    alpham = 0.32/mV * (vm + 54*mV) / (1.0 - exp(-(vm + 54.0*mV) / (4.0*mV)))/ms:Hz
    alphan = 0.032/mV * (vm + 52*mV) / (1.0 - exp(-(vm + 52.0*mV) / (5.0*mV)))/ms:Hz

    betah  = 4.0 / (1.0 + exp(-(vm + 27.0*mV) / (5.0*mV)))/ms:Hz
    betam  = 0.28/mV * (vm + 27.0*mV) / (exp((vm + 27.0*mV) / (5.0*mV)) - 1.0)/ms:Hz
    betan  = 0.5 * exp(-(vm + 57.0*mV) / (40.0*mV))/ms:Hz

    m = alpham / (alpham + betam) : 1
    membrane_Im = I_e + Isyn + Iinh + gNa_e*m**3*h*(ENa_e-vm) + \
        gl_e*(El_e-vm) + gK_e*n**4*(EK_e-vm) : amp

    dn/dt = alphan*(1-n)-betan*n : 1
    dh/dt = alphah*(1-h)-betah*h : 1
    dq/dt = 5/ms*(1+tanh(vm/(10*mV)))*(1-q) - q/tau_dq_e : 1
    ds/dt = q*(1-s)/tau_r_e - s/tau_d_e : 1
    dvm/dt = membrane_Im/C : volt
    """

    e_namespace = {"tau_r_e": tau_r_e, "tau_d_e": tau_d_e, "tau_dq_e": tau_dq_e,
                   "gl_e": gl_e, "gK_e": gK_e, "gNa_e": gNa_e, "El_e": El_e,
                   "EK_e": EK_e, "ENa_e": ENa_e, "alpha": alpha, "P": P,
                   "g_ex": g_ex, "v_rev_i": v_rev_i}

    e_cells = b2.NeuronGroup(num_e, eqs_e, threshold="vm>-20*mV", reset="", refractory=2*b2.ms,
                             method="rk4", dt=dt, namespace=e_namespace)
    e_cells.vm = np.asarray(IC_V[:num_e]) * b2.mV
    e_cells.h = np.asarray(IC_H[:num_e])
    e_cells.n = np.asarray(IC_N[:num_e])
    i_ext_e = i_ext_e_mean * (1 + sigma_e * rng.standard_normal(num_e)) * b2.uA
    if mode == "drive":
        e_cells.i_ext_e = i_ext_e
    else:
        e_cells.I_e = i_ext_e
    e_cells.q = 0
    e_cells.s = 0

    # --- I population (WB, phi=5 speed-up on h/n) ---
    El_i, EK_i, ENa_i = -65 * b2.mV, -90 * b2.mV, 55 * b2.mV
    gl_i, gK_i, gNa_i = 0.1 * b2.msiemens, 9 * b2.msiemens, 35 * b2.msiemens
    phi = 5.0

    eqs_i = """
    I_e : amp
    Isyn_ei : amp
    Isyn_ii : amp
    Isyn = Isyn_ei + Isyn_ii : amp

    alphah = 0.07 * exp(-(vm + 58.0*mV) / (20.0*mV))/ms :Hz
    alpham = 0.1/mV * (vm + 35.0*mV) / (1.0 - exp(-0.1/mV * (vm + 35.0*mV))) /ms :Hz
    alphan = -0.01/mV * (vm + 34.0*mV) / (exp(-0.1/mV * (vm + 34.0*mV)) - 1.0)/ms :Hz

    betah = 1.0 / (exp(-0.1/mV * (vm + 28.0*mV)) + 1.0)/ms :Hz
    betam = 4.0 * exp(-(vm + 60.0*mV) / (18.0*mV))/ms :Hz
    betan = 0.125 * exp(-(vm + 44.0*mV) / (80.0*mV))/ms :Hz

    m = alpham / (alpham + betam) : 1
    membrane_Im = I_e + Isyn + gNa_i*m**3*h*(ENa_i-vm) + \
        gl_i*(El_i-vm) + gK_i*n**4*(EK_i-vm) : amp

    dn/dt = phi*(alphan*(1-n)-betan*n) : 1
    dh/dt = phi*(alphah*(1-h)-betah*h) : 1
    dq/dt = 5/ms*(1+tanh(vm/(10*mV)))*(1-q) - q/tau_dq_i : 1
    ds/dt = q*(1-s)/tau_r_i - s/tau_d_i : 1
    dvm/dt = membrane_Im/C : volt
    """
    i_namespace = {"tau_r_i": tau_r_i, "tau_d_i": tau_d_i, "tau_dq_i": tau_dq_i,
                   "gl_i": gl_i, "gK_i": gK_i, "gNa_i": gNa_i, "El_i": El_i,
                   "EK_i": EK_i, "ENa_i": ENa_i, "phi": phi}
    i_cells = b2.NeuronGroup(num_i, eqs_i, threshold="vm>-20*mV", reset="", refractory=2*b2.ms,
                             method="rk4", dt=dt, namespace=i_namespace)
    i_cells.vm = -75 * b2.mV
    i_cells.h = "alphah / (alphah + betah)"
    i_cells.n = "alphan / (alphan + betan)"
    i_cells.I_e = 0 * b2.uA
    i_cells.q = 0
    i_cells.s = 0

    # --- synapses (deterministic all-to-all: p=1 for every connection
    # type in the book's script, so the "random connectivity" reduces to
    # uniform weight g_hat/N_pop) ---
    ei = b2.Synapses(e_cells, i_cells, "Isyn_ei_post = s_pre*(v_rev_e - vm_post)*g_hat_ei*msiemens/num_e : amp (summed)",
                      namespace={"v_rev_e": v_rev_e, "g_hat_ei": g_hat_ei, "num_e": num_e})
    ei.connect()
    ie = b2.Synapses(i_cells, e_cells, "Isyn_post = s_pre*(v_rev_i - vm_post)*g_hat_ie*msiemens/num_i : amp (summed)",
                      namespace={"v_rev_i": v_rev_i, "g_hat_ie": g_hat_ie, "num_i": num_i})
    ie.connect()
    ii = b2.Synapses(i_cells, i_cells, "Isyn_ii_post = s_pre*(v_rev_i - vm_post)*g_hat_ii*msiemens/num_i : amp (summed)",
                      namespace={"v_rev_i": v_rev_i, "g_hat_ii": g_hat_ii, "num_i": num_i})
    ii.connect(condition="i!=j")

    sp_mon_e = b2.SpikeMonitor(e_cells)
    sp_mon_i = b2.SpikeMonitor(i_cells)
    st_mon_e = b2.StateMonitor(e_cells, "vm", record=True)
    net = b2.Network(e_cells, i_cells, ei, ie, ii, sp_mon_e, sp_mon_i, st_mon_e)
    net.run(simulation_time)
    return sp_mon_e, sp_mon_i, st_mon_e

### Figure 34.11 / 34.12
PING Network Rastergram, Theta-Modulated Drive vs. Theta-Modulated Inhibition

In [ ]:
spm_e_drive, spm_i_drive, stm_e_drive = simulate_PING_with_theta("drive", 1000 * b2.ms)

fig, ax = plt.subplots(2, figsize=(10, 6), sharex=True)
ax[0].plot(spm_e_drive.t / b2.ms, spm_e_drive.i, ".r", ms=3)
ax[0].plot(spm_i_drive.t / b2.ms, spm_i_drive.i + 40, ".b", ms=3)
ax[0].set_ylabel("cell #")
ax[0].set_title("theta via drive modulation")
lfp = np.mean(stm_e_drive.vm[:] / b2.mV, axis=0)
ax[1].plot(stm_e_drive.t / b2.ms, lfp, "-k", lw=1)
ax[1].set_xlabel("t [ms]")
ax[1].set_ylabel("mean E-cell v [mV]")
plt.tight_layout()
plt.show()

In [ ]:
spm_e_inh, spm_i_inh, stm_e_inh = simulate_PING_with_theta("inhibition", 1000 * b2.ms)

fig, ax = plt.subplots(2, figsize=(10, 6), sharex=True)
ax[0].plot(spm_e_inh.t / b2.ms, spm_e_inh.i, ".r", ms=3)
ax[0].plot(spm_i_inh.t / b2.ms, spm_i_inh.i + 40, ".b", ms=3)
ax[0].set_ylabel("cell #")
ax[0].set_title("theta via periodic inhibition")
lfp_inh = np.mean(stm_e_inh.vm[:] / b2.mV, axis=0)
ax[1].plot(stm_e_inh.t / b2.ms, lfp_inh, "-k", lw=1)
ax[1].set_xlabel("t [ms]")
ax[1].set_ylabel("mean E-cell v [mV]")
plt.tight_layout()
plt.show()

### EIO_1: E, I, and OLM Populations Together

10 RTM E-cells, 5 WB I-cells, 5 OLM O-cells (h- and A-currents,
g_h=12, g_A=22). Coupling: E->I, I->E, I->I, I->O, O->E, O->I (E->O and
O->O are zero in the book's script). O-cells get a *negative* mean
drive (i_ext_o=-2.0, net inhibited) -- they only fire via
post-inhibitory rebound once I-cells' feedback lets go, which is the
"nested theta" mechanism this chapter is about.

The book's own script runs 200/50/50 cells; the Python port already
reduced that to 10/5/5 for tractability (still all-to-all coupling
normalized by population size, so the qualitative dynamics carry over).
Initial conditions come from the Python port's own `rtmInit`/`wbInit`/
`olmInit` (same warm-up idea as chapter 24's splay states), called once
here with a fixed seed and hard-coded below.

In [ ]:
IEXT_E = [1.8001107138021732, 1.8268870983757624, 1.7753275930174004, 1.7198467345118456, 1.759079629334545, 1.7107518100503185, 1.8054129242337695, 1.920619372099908, 1.7557014133303803, 1.7441572590162053]
IEXT_I = [1.0489842050185199, 1.035688700816006, 1.01054142489979, 0.9069531955291795, 0.9970748177536727]
IEXT_O = [-2.0695303194458288, -1.8655785452714917, -1.9542384238959782, -1.8098777260199155, -1.8710462260215024]
IV_E = [[-86.37985991708233, 0.9919065072303515, 0.01224149227526617], [-89.79326255904303, 0.9790204735702678, 0.031009639372990984], [-68.96327111635006, 0.9987628577272322, 0.01798542707623387], [-97.54284991320236, 0.7033371059970132, 0.2971615979572235], [-97.90697868951476, 0.6151038392511148, 0.3621990893480107], [-72.11933846832441, 0.9992148079835358, 0.010269937858519757], [-74.2054449640519, 0.9991753294161099, 0.007089651904993], [-59.658776137835545, 0.9918246645435373, 0.07317449396843818], [-68.44208534087058, 0.9986357838380039, 0.01980949319592194], [-72.23720401824531, 0.9992042668328773, 0.01001639497063242]]  # columns: v, h, n
IV_I = [[-60.66194960560875, 0.7246137401249855, 0.10655762144386018], [-64.7271868331108, 0.7533912248458006, 0.09672334826995871], [-43.953419177861335, 0.03181666729043875, 0.6866374593687837], [-65.61524995347642, 0.7341062675551694, 0.10526703451476684], [-57.60220735671403, 0.6347227165906563, 0.1322224826547075]]  # columns: v, h, n
IV_O = [[-68.79082351509244, 0.8082986617583953, 0.097136743897606, 0.014611336736158442, 0.03424007385015068, 0.2553231111398102], [-66.15778688218587, 0.7213169769570411, 0.10694136272403841, 0.020875433032497427, 0.04047679167840718, 0.29639865241806446], [-74.76876048952727, 0.09614801389221406, 0.4839710581726897, 0.00031688902798463446, 0.25801106887462216, 0.15649537968360444], [-59.340540939199755, 0.49527058657868006, 0.13553745979544327, 0.03414872495188927, 0.0593597389562131, 0.24000922534360578], [-69.56669441513566, 0.8508478595351168, 0.09279830413123005, 0.012275799770947995, 0.03131211196179649, 0.23466272225057205]]  # columns: v, h, n, r, a, b


def simulate_EIO(simulation_time, num_e=10, num_i=5, num_o=5,
                  g_hat_ei=0.25, g_hat_ie=0.25, g_hat_ii=0.25, g_hat_io=0.50,
                  g_hat_oe=1.00, g_hat_oi=0.50,
                  tau_r_e=0.5 * b2.ms, tau_d_e=3 * b2.ms, tau_dq_e=0.17229 * b2.ms,
                  tau_r_i=0.5 * b2.ms, tau_d_i=9 * b2.ms, tau_dq_i=0.11629 * b2.ms,
                  tau_r_o=0.5 * b2.ms, tau_d_o=20 * b2.ms, tau_dq_o=0.09453 * b2.ms,
                  g_h=12 * b2.msiemens, g_A=22 * b2.msiemens, dt=0.01 * b2.ms):
    v_rev_e, v_rev_i, v_rev_o = 0 * b2.mV, -75 * b2.mV, -75 * b2.mV

    # --- E population (RTM) ---
    eqs_e = """
    I_e : amp
    Isyn_ie : amp
    Isyn_oe : amp
    Isyn = Isyn_ie + Isyn_oe : amp

    alphah = 0.128 * exp(-(vm + 50.0*mV) / (18.0*mV))/ms :Hz
    alpham = 0.32/mV * (vm + 54*mV) / (1.0 - exp(-(vm + 54.0*mV) / (4.0*mV)))/ms:Hz
    alphan = 0.032/mV * (vm + 52*mV) / (1.0 - exp(-(vm + 52.0*mV) / (5.0*mV)))/ms:Hz

    betah  = 4.0 / (1.0 + exp(-(vm + 27.0*mV) / (5.0*mV)))/ms:Hz
    betam  = 0.28/mV * (vm + 27.0*mV) / (exp((vm + 27.0*mV) / (5.0*mV)) - 1.0)/ms:Hz
    betan  = 0.5 * exp(-(vm + 57.0*mV) / (40.0*mV))/ms:Hz

    m = alpham / (alpham + betam) : 1
    membrane_Im = I_e + Isyn + gNa_e*m**3*h*(ENa_e-vm) + \
        gl_e*(El_e-vm) + gK_e*n**4*(EK_e-vm) : amp

    dn/dt = alphan*(1-n)-betan*n : 1
    dh/dt = alphah*(1-h)-betah*h : 1
    dq/dt = 5/ms*(1+tanh(vm/(10*mV)))*(1-q) - q/tau_dq_e : 1
    ds/dt = q*(1-s)/tau_r_e - s/tau_d_e : 1
    dvm/dt = membrane_Im/C : volt
    """
    e_ns = {"tau_r_e": tau_r_e, "tau_d_e": tau_d_e, "tau_dq_e": tau_dq_e,
            "gl_e": 0.1 * b2.msiemens, "gK_e": 80 * b2.msiemens, "gNa_e": 100 * b2.msiemens,
            "El_e": -67 * b2.mV, "EK_e": -100 * b2.mV, "ENa_e": 50 * b2.mV, "C": 1 * b2.ufarad}
    e_cells = b2.NeuronGroup(num_e, eqs_e, threshold="vm>-20*mV", reset="", refractory=2 * b2.ms,
                             method="rk4", dt=dt, namespace=e_ns)
    iv_e = np.asarray(IV_E)
    e_cells.vm = iv_e[:, 0] * b2.mV
    e_cells.h = iv_e[:, 1]
    e_cells.n = iv_e[:, 2]
    e_cells.I_e = np.asarray(IEXT_E) * b2.uA
    e_cells.q = 0
    e_cells.s = 0

    # --- I population (WB, phi=5) ---
    eqs_i = """
    I_e : amp
    Isyn_ei : amp
    Isyn_ii : amp
    Isyn_oi : amp
    Isyn = Isyn_ei + Isyn_ii + Isyn_oi : amp

    alphah = 0.07 * exp(-(vm + 58.0*mV) / (20.0*mV))/ms :Hz
    alpham = 0.1/mV * (vm + 35.0*mV) / (1.0 - exp(-0.1/mV * (vm + 35.0*mV))) /ms :Hz
    alphan = -0.01/mV * (vm + 34.0*mV) / (exp(-0.1/mV * (vm + 34.0*mV)) - 1.0)/ms :Hz

    betah = 1.0 / (exp(-0.1/mV * (vm + 28.0*mV)) + 1.0)/ms :Hz
    betam = 4.0 * exp(-(vm + 60.0*mV) / (18.0*mV))/ms :Hz
    betan = 0.125 * exp(-(vm + 44.0*mV) / (80.0*mV))/ms :Hz

    m = alpham / (alpham + betam) : 1
    membrane_Im = I_e + Isyn + gNa_i*m**3*h*(ENa_i-vm) + \
        gl_i*(El_i-vm) + gK_i*n**4*(EK_i-vm) : amp

    dn/dt = phi*(alphan*(1-n)-betan*n) : 1
    dh/dt = phi*(alphah*(1-h)-betah*h) : 1
    dq/dt = 5/ms*(1+tanh(vm/(10*mV)))*(1-q) - q/tau_dq_i : 1
    ds/dt = q*(1-s)/tau_r_i - s/tau_d_i : 1
    dvm/dt = membrane_Im/C : volt
    """
    i_ns = {"tau_r_i": tau_r_i, "tau_d_i": tau_d_i, "tau_dq_i": tau_dq_i,
            "gl_i": 0.1 * b2.msiemens, "gK_i": 9 * b2.msiemens, "gNa_i": 35 * b2.msiemens,
            "El_i": -65 * b2.mV, "EK_i": -90 * b2.mV, "ENa_i": 55 * b2.mV,
            "phi": 5.0, "C": 1 * b2.ufarad}
    i_cells = b2.NeuronGroup(num_i, eqs_i, threshold="vm>-20*mV", reset="", refractory=2 * b2.ms,
                             method="rk4", dt=dt, namespace=i_ns)
    iv_i = np.asarray(IV_I)
    i_cells.vm = iv_i[:, 0] * b2.mV
    i_cells.h = iv_i[:, 1]
    i_cells.n = iv_i[:, 2]
    i_cells.I_e = np.asarray(IEXT_I) * b2.uA
    i_cells.q = 0
    i_cells.s = 0

    # --- O population (OLM: h- and A-currents; c=1.3, unlike E/I's c=1) ---
    eqs_o = """
    I_e : amp
    Isyn : amp

    alpham = (vm + 38*mV) / (10*mV) / (1.0 - exp(-(vm + 38*mV) / (10*mV))) /ms : Hz
    alphah = 0.07 * exp(-(vm + 63.0*mV) / (20.0*mV))/ms : Hz
    alphan = 0.018/mV * (vm - 25*mV) / (1.0 - exp(-(vm - 25*mV) / (25*mV)))/ms : Hz

    betam = 4.0 * exp(-(vm + 65.0*mV) / (18.0*mV))/ms : Hz
    betah = 1.0 / (exp(-(vm + 33.0*mV) / (10.0*mV)) + 1.0)/ms : Hz
    betan = 0.0036/mV * (35*mV - vm) / (1.0 - exp(-(35*mV - vm) / (12*mV)))/ms : Hz

    r_inf = 1.0 / (1.0 + exp((vm + 84.0*mV) / (10.2*mV))) : 1
    tau_r = 1/(exp(-14.59 - 0.086*vm/mV) + exp(-1.87 + 0.0701*vm/mV))*ms : second

    a_inf = 1.0 / (1.0 + exp(-(vm + 14.0*mV) / (16.6*mV))) : 1
    b_inf = 1.0 / (1.0 + exp((vm + 71.0*mV) / (7.3*mV))) : 1
    tau_a = 5*ms : second
    tau_b = 1/(0.000009/exp((vm/mV - 26)/28.5) + 0.014/(0.2 + exp(-(vm/mV + 70.0)/11.0)))*ms : second

    m = alpham / (alpham + betam) : 1
    membrane_Im = (I_e + Isyn + gNa_o*m**3*h*(ENa_o-vm) + gl_o*(El_o-vm) + gK_o*n**4*(EK_o-vm)
        + g_h*r*(EH_o-vm) + g_A*a*b*(EA_o-vm)) / 1.3 : amp

    dn/dt = alphan*(1-n)-betan*n : 1
    dh/dt = alphah*(1-h)-betah*h : 1
    dr/dt = (r_inf - r) / tau_r : 1
    da/dt = (a_inf - a) / tau_a : 1
    db/dt = (b_inf - b) / tau_b : 1
    dq/dt = 10/ms*(0.5*(1+tanh(vm/(10*mV))))*(1-q) - q/tau_dq_o : 1
    ds/dt = q*(1-s)/tau_r_o - s/tau_d_o : 1
    dvm/dt = membrane_Im/C : volt
    """
    o_ns = {"tau_r_o": tau_r_o, "tau_d_o": tau_d_o, "tau_dq_o": tau_dq_o, "g_h": g_h, "g_A": g_A,
            "gl_o": 0.05 * b2.msiemens, "gK_o": 23 * b2.msiemens, "gNa_o": 30 * b2.msiemens,
            "El_o": -70 * b2.mV, "EK_o": -100 * b2.mV, "ENa_o": 90 * b2.mV,
            "EH_o": -32.9 * b2.mV, "EA_o": -90 * b2.mV, "C": 1 * b2.ufarad}
    o_cells = b2.NeuronGroup(num_o, eqs_o, threshold="vm>-20*mV", reset="", refractory=2 * b2.ms,
                             method="rk4", dt=dt, namespace=o_ns)
    iv_o = np.asarray(IV_O)
    o_cells.vm = iv_o[:, 0] * b2.mV
    o_cells.h = iv_o[:, 1]
    o_cells.n = iv_o[:, 2]
    o_cells.r = iv_o[:, 3]
    o_cells.a = iv_o[:, 4]
    o_cells.b = iv_o[:, 5]
    o_cells.I_e = np.asarray(IEXT_O) * b2.uA
    o_cells.q = 0
    o_cells.s = 0

    # --- synapses (p=1 for every connection type, so uniform weight g_hat/N_pop) ---
    ei = b2.Synapses(e_cells, i_cells, "Isyn_ei_post = s_pre*(v_rev_e - vm_post)*g_hat_ei*msiemens/num_e : amp (summed)",
                      namespace={"v_rev_e": v_rev_e, "g_hat_ei": g_hat_ei, "num_e": num_e})
    ei.connect()
    ie = b2.Synapses(i_cells, e_cells, "Isyn_ie_post = s_pre*(v_rev_i - vm_post)*g_hat_ie*msiemens/num_i : amp (summed)",
                      namespace={"v_rev_i": v_rev_i, "g_hat_ie": g_hat_ie, "num_i": num_i})
    ie.connect()
    ii = b2.Synapses(i_cells, i_cells, "Isyn_ii_post = s_pre*(v_rev_i - vm_post)*g_hat_ii*msiemens/num_i : amp (summed)",
                      namespace={"v_rev_i": v_rev_i, "g_hat_ii": g_hat_ii, "num_i": num_i})
    ii.connect(condition="i!=j")
    io = b2.Synapses(i_cells, o_cells, "Isyn_post = s_pre*(v_rev_i - vm_post)*g_hat_io*msiemens/num_i : amp (summed)",
                      namespace={"v_rev_i": v_rev_i, "g_hat_io": g_hat_io, "num_i": num_i})
    io.connect()
    oe = b2.Synapses(o_cells, e_cells, "Isyn_oe_post = s_pre*(v_rev_o - vm_post)*g_hat_oe*msiemens/num_o : amp (summed)",
                      namespace={"v_rev_o": v_rev_o, "g_hat_oe": g_hat_oe, "num_o": num_o})
    oe.connect()
    oi = b2.Synapses(o_cells, i_cells, "Isyn_oi_post = s_pre*(v_rev_o - vm_post)*g_hat_oi*msiemens/num_o : amp (summed)",
                      namespace={"v_rev_o": v_rev_o, "g_hat_oi": g_hat_oi, "num_o": num_o})
    oi.connect()

    sp_e = b2.SpikeMonitor(e_cells)
    sp_i = b2.SpikeMonitor(i_cells)
    sp_o = b2.SpikeMonitor(o_cells)
    net = b2.Network(e_cells, i_cells, o_cells, ei, ie, ii, io, oe, oi, sp_e, sp_i, sp_o)
    net.run(simulation_time)
    return sp_e, sp_i, sp_o

### Figure 34.14
EIO_1 Rastergram (E, I, O populations)

In [ ]:
spm_e, spm_i, spm_o = simulate_EIO(1000 * b2.ms)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(spm_e.t / b2.ms, spm_e.i, ".r", ms=4, label="E")
ax.plot(spm_i.t / b2.ms, spm_i.i + 10, ".b", ms=4, label="I")
ax.plot(spm_o.t / b2.ms, spm_o.i + 15, ".g", ms=4, label="O")
ax.set_xlabel("t [ms]")
ax.set_ylabel("cell #")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()